# FundamenTracker Function Tests in Google Colab
This notebook prepares an import-safe copy of `src/main.py`, then tests its core functions using standard `unittest` cases.

## Section 1: Import Required Libraries

In [ ]:
import unittest
import os
import sys
import json
import tempfile
import shutil
from pathlib import Path
import importlib.util

try:
    from google.colab import drive
    COLAB_AVAILABLE = True
except ImportError:
    drive = None
    COLAB_AVAILABLE = False

print('Colab available:', COLAB_AVAILABLE)
print('Python version:', sys.version)


## Section 2: Mount Google Drive
This section mounts Google Drive for optional repository access and file sharing.

## Section 3: Verify Colab Runtime
This section checks the runtime environment, current directory, and available files.

## Section 4: Load or Define Functions
This section loads `src/main.py` into an import-safe temporary module so its functions can be tested without executing the full script.

## Section 5: Write Unit Tests
This section defines unit test cases for state handling and Telegram command parsing.

## Section 6: Run Tests and Display Results
This section executes the `unittest` suite and prints a summary of the results.

## Section 7: Cleanup and Unmount Drive
This section removes temporary files and unmounts Google Drive if it was mounted.

In [ ]:
# Mount Google Drive (optional)
# If running in Colab, this will mount Google Drive so you can access repository files from Drive.
if COLAB_AVAILABLE:
    try:
        drive.mount('/content/drive', force_remount=False)
        print('Mounted Google Drive successfully.')
    except Exception as e:
        print('Drive mount failed:', e)
else:
    print('Google Drive mount not available in this environment.')


In [ ]:
# Verify Colab runtime environment
import platform

print('Platform:', platform.platform())
print('Current working directory:', os.getcwd())
print('Files in current directory:', os.listdir('.'))


In [ ]:
# Load or define functions from src/main.py in an import-safe temporary module
repo_root = Path('.')
source_file = repo_root / 'src' / 'main.py'
print('Source file exists:', source_file.exists())

if not source_file.exists():
    raise FileNotFoundError(f'Could not find {source_file}')

raw_code = source_file.read_text()
lines = raw_code.splitlines()

# Wrap the top-level execution section to make the module safe to import.
start_index = None
for idx, line in enumerate(lines):
    if line.strip() == '# PROCESS TELEGRAM COMMANDS':
        start_index = idx
        break

if start_index is None:
    raise RuntimeError('Could not locate the main execution section in src/main.py')

wrapped_lines = lines[:start_index]
wrapped_lines.append('if __name__ == "__main__":')
wrapped_lines.extend('    ' + line if line.strip() != '' else '' for line in lines[start_index:])

temp_module_path = repo_root / 'tmp_main_for_tests.py'
temp_module_path.write_text('\n'.join(wrapped_lines) + '\n')
print('Created temporary import-safe module at', temp_module_path)

spec = importlib.util.spec_from_file_location('tmp_main_for_tests', str(temp_module_path))
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print('Imported module attributes:', [a for a in dir(mod) if a in ['load_state','save_state','process_telegram_commands','send_message','get_updates']])


In [ ]:
# Write unit tests for the main module functions
class TestStateHandling(unittest.TestCase):
    def setUp(self):
        self.temp_dir = tempfile.TemporaryDirectory()
        self.original_state_file = mod.STATE_FILE
        mod.STATE_FILE = str(Path(self.temp_dir.name) / 'state.json')

    def tearDown(self):
        mod.STATE_FILE = self.original_state_file
        self.temp_dir.cleanup()

    def test_load_state_missing_file(self):
        if os.path.exists(mod.STATE_FILE):
            os.remove(mod.STATE_FILE)
        state = mod.load_state()
        self.assertEqual(state, {'watchlist': {}, 'last_update_id': 0})

    def test_save_and_load_state(self):
        test_state = {'watchlist': {'AAPL': {'name': 'Apple', 'pe_trigger': 15.0, 'last_pe_alert': None}}, 'last_update_id': 42}
        mod.save_state(test_state)
        loaded = mod.load_state()
        self.assertEqual(loaded, test_state)

    def test_load_state_corrupt_file(self):
        Path(mod.STATE_FILE).write_text('not a json')
        state = mod.load_state()
        self.assertEqual(state, {'watchlist': {}, 'last_update_id': 0})

class TestCommandParser(unittest.TestCase):
    def setUp(self):
        self.temp_dir = tempfile.TemporaryDirectory()
        self.original_state_file = mod.STATE_FILE
        mod.STATE_FILE = str(Path(self.temp_dir.name) / 'state.json')
        mod.save_state({'watchlist': {}, 'last_update_id': 0})
        self.sent_messages = []

        def fake_send_message(text):
            self.sent_messages.append(text)

        self.original_send_message = mod.send_message
        self.original_get_updates = mod.get_updates
        mod.send_message = fake_send_message

    def tearDown(self):
        mod.STATE_FILE = self.original_state_file
        mod.send_message = self.original_send_message
        mod.get_updates = self.original_get_updates
        self.temp_dir.cleanup()

    def test_add_command(self):
        sample_update = {
            'update_id': 1,
            'message': {'text': '/add AAPL 15'}
        }
        mod.get_updates = lambda last_update_id: [sample_update]
        state = {'watchlist': {}, 'last_update_id': 0}
        new_state = mod.process_telegram_commands(state)
        self.assertIn('AAPL', new_state['watchlist'])
        self.assertEqual(new_state['watchlist']['AAPL']['pe_trigger'], 15.0)
        self.assertIn('✅ Added', self.sent_messages[0])

    def test_remove_command(self):
        state = {'watchlist': {'AAPL': {'name': 'Apple', 'pe_trigger': 15.0, 'last_pe_alert': None}}, 'last_update_id': 0}
        sample_update = {
            'update_id': 2,
            'message': {'text': '/remove AAPL'}
        }
        mod.get_updates = lambda last_update_id: [sample_update]
        new_state = mod.process_telegram_commands(state)
        self.assertNotIn('AAPL', new_state['watchlist'])
        self.assertIn('🗑 Removed', self.sent_messages[0])

    def test_list_command(self):
        state = {'watchlist': {'AAPL': {'name': 'Apple', 'pe_trigger': 15.0, 'last_pe_alert': None}}, 'last_update_id': 0}
        sample_update = {
            'update_id': 3,
            'message': {'text': '/list'}
        }
        mod.get_updates = lambda last_update_id: [sample_update]
        new_state = mod.process_telegram_commands(state)
        self.assertEqual(new_state['watchlist']['AAPL']['name'], 'Apple')
        self.assertTrue(any('Current watchlist' in msg for msg in self.sent_messages))

    def test_resetstate_command(self):
        state = {'watchlist': {'AAPL': {'name': 'Apple', 'pe_trigger': 15.0, 'last_pe_alert': None}}, 'last_update_id': 0}
        sample_update = {
            'update_id': 4,
            'message': {'text': '/resetstate'}
        }
        mod.get_updates = lambda last_update_id: [sample_update]
        new_state = mod.process_telegram_commands(state)
        self.assertEqual(new_state['watchlist'], {})
        self.assertEqual(new_state['last_update_id'], 4)
        self.assertTrue(any('State reset' in msg for msg in self.sent_messages))


In [ ]:
# Run tests and display results
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestStateHandling)
suite.addTests(loader.loadTestsFromTestCase(TestCommandParser))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print('Tests run:', result.testsRun)
print('Failures:', len(result.failures))
print('Errors:', len(result.errors))


In [ ]:
# Cleanup and optional unmount
try:
    if temp_module_path.exists():
        temp_module_path.unlink()
        print('Cleaned up temporary test module.')
except NameError:
    print('Temporary module path not defined, nothing to clean.')

if COLAB_AVAILABLE:
    try:
        drive.flush_and_unmount()
        print('Unmounted Google Drive.')
    except Exception as e:
        print('Could not unmount Drive:', e)
else:
    print('No Google Drive unmount needed.')
